# ANAADHI — Kaggle GPU Movie Shot Generator

GitHub-managed, Android-friendly notebook. Do not manually edit code unless asked.

**Current target:** `SC001_SH001` — raised cabin establishing shot.

Use a **Tesla T4** accelerator. P100 is intentionally rejected because the current Kaggle/PyTorch stack can fail on it.

In [ ]:
import sys, subprocess, importlib.util

required = ['diffusers', 'transformers', 'accelerate', 'safetensors', 'peft']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Required AI packages already available — skipping pip download.')

import torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from IPython.display import display

assert torch.cuda.is_available(), 'GPU is not enabled. Kaggle Settings → Accelerator → GPU T4.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
print('CUDA:', torch.version.cuda)
if 'P100' in GPU_NAME.upper():
    raise RuntimeError('P100 is not supported by this notebook. Switch Kaggle Accelerator to GPU T4 and rerun.')


## Locked shot configuration

This cell is maintained from GitHub. It uses a short prompt so SDXL CLIP does not truncate the important cabin instructions.

In [ ]:
SHOT_ID = 'SC001_SH001'
MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'photoreal cinematic anamorphic wide shot, large raised rough timber cabin dominant in center foreground, pre-dawn Karnataka Western Ghats, black monsoon rain, wet giant roots, laterite mud, mist behind cabin, black-kite police drones overhead, camouflaged medical transport under areca leaves, distant tactical silhouettes, dark psychological thriller'
NEGATIVE_PROMPT = 'empty forest, missing cabin, tiny cabin, cabin hidden by trees, daylight, sunny sky, cyberpunk, American police cars, firefight, explosion, fantasy, anime, illustration, text, watermark, subtitles, black bars, blurry'
SEED = 1002
STEPS = 32
GUIDANCE = 6.5
GEN_WIDTH = 1344
GEN_HEIGHT = 640
MASTER_WIDTH = 3840
MASTER_HEIGHT = 1608
OUTPUT_DIR = Path('/kaggle/working/anaadhi_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Shot:', SHOT_ID)


## Load SDXL

The model stays memory-safe using CPU offload. The first load is the slow part; later shot generations are much faster.

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print('Model ready:', MODEL_ID)


## Generate SC001_SH001

This saves a clean scope image without baked black bars.

In [ ]:
generator = torch.Generator(device='cpu').manual_seed(SEED)
image = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=GEN_WIDTH,
    height=GEN_HEIGHT,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    generator=generator,
).images[0]

target_ratio = MASTER_WIDTH / MASTER_HEIGHT
new_h = round(image.width / target_ratio)
if new_h <= image.height:
    y0 = (image.height - new_h) // 2
    scoped = image.crop((0, y0, image.width, y0 + new_h))
else:
    new_w = round(image.height * target_ratio)
    x0 = (image.width - new_w) // 2
    scoped = image.crop((x0, 0, x0 + new_w, image.height))

out_path = OUTPUT_DIR / f'{SHOT_ID}_seed{SEED}.png'
scoped.save(out_path)
print('Saved:', out_path)
print('Working output:', scoped.size, 'ratio:', round(scoped.width / scoped.height, 4))
display(scoped)


## Next

If the image is not approved, send the result to ChatGPT. The GitHub notebook will be revised there; you do not need to repair Python by hand.